# Phase 1, Step 1: Data Ingestion and Exploration

This notebook covers the initial phase of the project, focusing on loading the support ticket data and performing Exploratory Data Analysis (EDA). The goal is to understand the dataset's structure, content, and characteristics before moving to modeling.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Ensure NLTK data for VADER is downloaded
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except nltk.downloader.DownloadError:
    nltk.download('vader_lexicon')

%matplotlib inline
sns.set(style="whitegrid")

## 2. Load the Dataset

Since we are dealing with 1 million tickets, we'll use `pd.read_csv` to load the data. If the file is extremely large and causes memory issues, you may need to load it in chunks, but for typical datasets of this size, a direct read should suffice. We'll specify a `dtype` for the text column to save memory.

In [ ]:
file_path = '../data/raw/support_tickets.csv'

# It's good practice to define column types for large files to optimize memory usage
try:
    df = pd.read_csv(file_path, dtype={'ticket_description': 'string'})
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors later

if not df.empty:
    print(f"Dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
    display(df.head())

## 3. Exploratory Data Analysis (EDA)

### Basic Information

In [ ]:
if not df.empty:
    df.info()
    df.describe(include='all')

### Missing Values

In [ ]:
if not df.empty:
    missing_values = df.isnull().sum()
    print("\nMissing values in each column:\n", missing_values[missing_values > 0])
    
    # Visualize missing data
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
    plt.title('Heatmap of Missing Values')
    plt.show()

### Ticket Volume Analysis

We can visualize the volume of tickets over time, by category, or by some other relevant dimension.

In [ ]:
if 'created_at' in df.columns and not df.empty:
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['creation_date'] = df['created_at'].dt.date
    
    ticket_volume = df.groupby('creation_date').size()
    
    plt.figure(figsize=(15, 7))
    ticket_volume.plot()
    plt.title('Daily Ticket Volume')
    plt.xlabel('Date')
    plt.ylabel('Number of Tickets')
    plt.show()

### Ticket Types and Distribution

If a column like `ticket_type` or `product` exists, we can analyze the distribution.

In [ ]:
if 'ticket_type' in df.columns and not df.empty:
    plt.figure(figsize=(12, 8))
    df['ticket_type'].value_counts().plot(kind='bar')
    plt.title('Ticket Type Distribution')
    plt.xlabel('Ticket Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

### Basic Sentiment Analysis

Using NLTK's VADER, we can get a quick sense of the sentiment in the ticket descriptions. This provides a baseline understanding of customer emotion.

In [ ]:
if 'ticket_description' in df.columns and not df.empty:
    sid = SentimentIntensityAnalyzer()
    
    # Sample the data for faster processing, as 1 million rows can be slow
    sample_df = df.sample(min(100000, len(df)), random_state=42)
    
    sample_df['sentiment'] = sample_df['ticket_description'].astype(str).apply(lambda x: sid.polarity_scores(x)['compound'])
    
    plt.figure(figsize=(10, 6))
    sns.histplot(sample_df['sentiment'], bins=20, kde=True)
    plt.title('Distribution of Ticket Sentiment')
    plt.xlabel('Sentiment Score (-1 to 1)')
    plt.show()
    
    # Categorize sentiment and show counts
    sample_df['sentiment_label'] = pd.cut(sample_df['sentiment'], bins=[-1, -0.05, 0.05, 1], labels=['Negative', 'Neutral', 'Positive'])
    print("\nSentiment Label Distribution:\n", sample_df['sentiment_label'].value_counts())


### Ticket Closure Times

This is a key metric for understanding operational efficiency. We'll need `created_at` and a `closed_at` column.

In [ ]:
if 'created_at' in df.columns and 'closed_at' in df.columns and not df.empty:
    df['closed_at'] = pd.to_datetime(df['closed_at'])
    df['closure_time_hours'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600
    
    plt.figure(figsize=(10, 6))
    sns.histplot(df['closure_time_hours'], bins=50, kde=True)
    plt.title('Distribution of Ticket Closure Times')
    plt.xlabel('Closure Time (Hours)')
    plt.ylabel('Count')
    plt.xlim(0, df['closure_time_hours'].quantile(0.95)) # Focus on the bulk of the data
    plt.show()

### Agent Routing Patterns

Assuming columns like `assigned_agent` or `assigned_team` exist, we can analyze routing.

In [ ]:
if 'assigned_team' in df.columns and not df.empty:
    plt.figure(figsize=(12, 8))
    df['assigned_team'].value_counts().plot(kind='bar')
    plt.title('Ticket Distribution per Team')
    plt.xlabel('Assigned Team')
    plt.ylabel('Number of Tickets')
    plt.xticks(rotation=45)
    plt.show()